# **Суперпозиция и SAE: практика**

В этой практике две части.

1. **Суперпозиция на игрушечной модели.** Повторим опыт [Elhage et al., 2022](https://transformer-circuits.pub/2022/toy_model/index.html): пять признаков сжимаются в два измерения, и по разреженности признаков видно, когда модель начинает хранить их в суперпозиции.
2. **SAE Lens.** Загрузим готовый разреженный автокодировщик для GPT-2 small, разберем его устройство и прочитаем признаки, которые он находит в остаточном потоке.

Задания пронумерованы, ответы на них вводятся на сайте курса в уроке «Суперпозиция и SAE: Практика».

## **Подготовка окружения**
Для этой практики хватит CPU: дополнительные вычислительные ресурсы не нужны.

In [ ]:
!pip install transformer-lens==3.9.0 plotly -q

In [ ]:
!pip install sae-lens==6.54.0 -q

****
**После выполнения ячеек выше перезапустите ноутбук, прежде чем приступить к заданиям.**
****

In [ ]:
# Стандартные библиотеки
import math
import torch
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

torch.set_grad_enabled(False)  # градиенты нужны только при обучении игрушечной модели, там они включаются явно

# MPS не выбираем: TransformerLens предупреждает, что на нем результаты могут быть неверны
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# **Часть 1. Суперпозиция на игрушечной модели**

У модели $n = 5$ признаков и всего $m = 2$ измерения. Признак $x_i$ с вероятностью $p$ равен случайному числу из $[0, 1]$, а иначе нулю; $1 - p$ называют разреженностью. Модель сжимает вход матрицей $W \in \mathbb{R}^{2 \times 5}$ и восстанавливает его:

$$\hat x = \mathrm{ReLU}(W^\top W x + b).$$

Ошибка восстановления взвешена важностью признаков $I_i = 0.9^i$: признак 0 важнее всех. Столбец $W_i$ — направление, которым модель записывает признак $i$ в двумерное пространство. Если длина столбца близка к 1, признак представлен; если к 0, модель его отбросила.

In [ ]:
N_FEATURES, N_DIMS = 5, 2
importance = 0.9 ** torch.arange(N_FEATURES)  # важность признака i равна 0.9^i


def train_toy(p, steps=5000, seed=0):
    """Обучает W (2 x 5) и b так, чтобы ReLU(W^T W x + b) восстанавливал x.

    p — вероятность, что признак ненулевой; 1 - p — разреженность.
    Модель крошечная и считается на CPU: так один и тот же сид дает одинаковый результат на любой машине.
    """
    g = torch.Generator().manual_seed(seed)
    W = torch.nn.Parameter(torch.randn(N_DIMS, N_FEATURES, generator=g) * 0.1)
    b = torch.nn.Parameter(torch.zeros(N_FEATURES))
    opt = torch.optim.AdamW([W, b], lr=1e-2, weight_decay=0)
    with torch.enable_grad():
        for _ in range(steps):
            x = torch.rand(1024, N_FEATURES, generator=g) * (torch.rand(1024, N_FEATURES, generator=g) < p)
            x_hat = torch.relu(x @ W.T @ W + b)
            loss = (importance * (x - x_hat) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return W.detach(), b.detach()

Обучим модель при пяти значениях $p$, от плотных признаков ($p = 1$) до очень разреженных ($p = 0.01$). Для каждого $p$ посчитайте, сколько признаков представлено (длина столбца больше 0.5), и сколько измерений приходится на один признак: $m / \|W\|_F^2$. Если признаков столько же, сколько измерений, и каждый занимает свое, это число равно 1; чем оно меньше, тем плотнее признаки упакованы.

In [ ]:
toy = {p: train_toy(p) for p in [1.0, 0.3, 0.1, 0.05, 0.01]}

for p, (W, b) in toy.items():
    norms = W.norm(dim=0)  # длина столбца W_i: насколько признак i представлен в двух измерениях
    represented = ...  # Ваш код здесь: число признаков с длиной столбца больше 0.5
    dims_per_feature = ...  # Ваш код здесь: N_DIMS / ||W||_F^2
    print(f"p={p}: длины {[round(v, 2) for v in norms.tolist()]}, представлено {represented}, измерений на признак {dims_per_feature}")

**Задание 1. Сколько признаков представлено при плотных признаках, $p = 1$?**

**Задание 2. Сколько признаков представлено при $p = 0.05$?**

**Задание 3. Сколько измерений приходится на один признак при $p = 0.05$? Ответ округлите до сотых.**

Нарисуем столбцы $W$ для плотного и разреженного случая и посмотрим на углы между ними и на смещения $b$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, p in zip(axes, [1.0, 0.05]):
    W, b = toy[p]
    for i in range(N_FEATURES):
        ax.plot([0, W[0, i]], [0, W[1, i]], marker="o")
        ax.annotate(f"{i}", (W[0, i] * 1.1, W[1, i] * 1.1))
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal")
    ax.set_title(f"p = {p}")
plt.show()

W, b = toy[0.05]
angles = sorted((torch.atan2(W[1], W[0]) * 180 / math.pi).tolist())
print("углы столбцов, градусы:", [round(a, 1) for a in angles])
print("смещения b при p=0.05:", [round(v, 3) for v in b.tolist()])
print("смещения b при p=1.0:", [round(v, 3) for v in toy[1.0][1].tolist()])

**Задание 4. Какой знак у смещений $b$ при $p = 0.05$ и зачем модели такой знак?**

При $p = 0.05$ пять признаков лежат в двух измерениях, поэтому направления не ортогональны: активный признак дает небольшую проекцию на чужие направления. Это и есть помехи суперпозиции. Подумайте, что с такими помехами делает ReLU вместе со смещением.

# **Часть 2. SAE Lens**

В игрушечной модели мы знали признаки заранее. В настоящей модели их не знает никто, и разреженный автокодировщик (SAE) пытается найти их сам: раскладывает активацию по словарю направлений, которых больше, чем измерений. Библиотека [SAE Lens](https://github.com/decoderesearch/SAELens) хранит каталог обученных SAE — для GPT-2, Gemma 2 (проект [Gemma Scope](https://huggingface.co/google/gemma-scope)), Llama и других моделей.

In [ ]:
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory

df = pd.DataFrame.from_records(
    {k: v.__dict__ for k, v in get_pretrained_saes_directory().items()}
).T.reset_index(drop=True)

df.head()

Устройство таблицы:

- `release` — релиз, то есть набор SAE, обученных одним подходом. Например, `gemma-scope-2b-pt-att`: `gemma-scope` — проект Gemma Scope, `2b-pt` — базовая модель Gemma 2 2B, `att` — SAE обучены на выходах attention-слоя;
- `repo_id` — репозиторий на Hugging Face, где лежат веса; там часто есть подробная карточка;
- `model` — модель, на активациях которой обучены SAE;
- `saes_map` — все SAE релиза: имя SAE и хук, к которому он подключается;
- `neuronpedia_id` — заполнено, если признаки SAE можно посмотреть на Neuronpedia.

Выведем SAE двух релизов для GPT-2 small: по одному SAE на остаточный поток каждого слоя и релиз, где на одном хуке обучены SAE разного размера.

In [ ]:
for release in ["gpt2-small-res-jb", "gpt2-small-res-jb-feature-splitting"]:
    print(f"SAE в релизе {release}:")
    for sae_id, hook in df.loc[df.release == release, "saes_map"].values[0].items():
        print(f"  {sae_id} -> {hook}")

n_saes = ...  # Ваш код здесь: число SAE в релизе gpt2-small-res-jb (длина его saes_map)
print("SAE в gpt2-small-res-jb:", n_saes)
print("релизов Gemma Scope в каталоге:", int(df.release.str.startswith("gemma-scope").sum()))

**Задание 5. Сколько SAE в релизе `gpt2-small-res-jb`?**

**Задание 6. Сколько разных SAE можно обучить на одной модели?** Сравните число SAE в двух релизах выше с числом слоев GPT-2 small (их 12).

## **Пара SAE и модель**

Загрузим GPT-2 small в обертке TransformerLens и SAE, обученный на остаточном потоке перед блоком 7 (`blocks.7.hook_resid_pre`). Вспомним, через какие точки проходит остаточный поток внутри блока:

`x_pre ──► Attention ──► x_mid ──► MLP ──► x_post (= x_pre следующего блока)`

In [ ]:
from sae_lens import SAE, HookedSAETransformer

model = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # release
    sae_id="blocks.7.hook_resid_pre",  # SAE id
    device=device,
)

In [ ]:
model.blocks[7]

По имени хука видно, какую точку блока он сохраняет.

**Задание 7. Сопоставьте хук и точку в модели:** `hook_normalized`, `hook_q_input`, `hook_resid_mid`, `hook_resid_post`.

## **Устройство SAE**

SAE — тоже модель с хуками. Конфигурация хранит размерность входа `d_in` и размер словаря `d_sae`.

In [ ]:
print(sae)
print(sae.cfg.to_dict())

expansion = ...  # Ваш код здесь: коэффициент расширения d_sae / d_in
print("коэффициент расширения:", expansion)

**Задание 8. Чему равен коэффициент расширения этого SAE?**

## **Признаки SAE на Neuronpedia**

Признак SAE — одна координата скрытого вектора, то есть одно направление словаря. Что он означает, определяют по текстам, на которых он сильнее всего активируется. На [Neuronpedia](https://www.neuronpedia.org/) для каждого признака есть такие примеры, графики логитов, на которые признак влияет, и описание, написанное языковой моделью. Ячейка ниже показывает дашборд признака 3777 (работает, если в окружении открывается внешний сайт).

In [ ]:
from IPython.display import IFrame

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"


def get_dashboard_html(sae_release="gpt2-small", sae_id="7-res-jb", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)


IFrame(get_dashboard_html(feature_idx=3777), width=1200, height=600)

Описания признаков можно выгрузить таблицей и искать признаки на заданную тему. Загрузим описания признаков нашего SAE.

In [ ]:
import gzip
import json
import re
import urllib.request

import requests


# Выгрузки разметки признаков Neuronpedia лежат в открытом S3-наборе neuronpedia-datasets (батчи jsonl.gz)
def load_explanations(model_id, sae_id):
    base = "https://neuronpedia-datasets.s3.us-east-1.amazonaws.com"
    listing = requests.get(base, params={"list-type": "2", "prefix": f"v1/{model_id}/{sae_id}/explanations/"}).text
    rows = []
    for key in re.findall(r"<Key>([^<]+\.jsonl\.gz)</Key>", listing):
        raw = gzip.decompress(urllib.request.urlopen(f"{base}/{key}").read()).decode()
        rows += [{k: v for k, v in json.loads(line).items() if k != "embedding"} for line in raw.splitlines()]
    return rows


explanations_df = pd.DataFrame(load_explanations("gpt2-small", "7-res-jb"))
explanations_df.rename(columns={"index": "feature"}, inplace=True)
explanations_df["description"] = explanations_df["description"].str.lower()
explanations_df[["feature", "description", "explanationModelName"]].head()

In [ ]:
fruits_features = ...  # Ваш код здесь: строки explanations_df, где в description есть " fruits"
fruits_features

**Задание 9. Сколько признаков этого SAE описаны словом «fruits»?**

Описания написала модель-аннотатор, а не человек, и признак с подходящим описанием еще не обязан срабатывать на всех фруктах: описание — гипотеза, которую проверяют на новых текстах.

## **Прогон модели с SAE**

Посмотрим, как модель продолжает промпт, а потом прогоним его вместе с SAE и сохраним все хуки в кэш.

In [ ]:
from transformer_lens.utils import test_prompt

prompt = "The best animal of the world is a"
answer = "cat"
test_prompt(prompt, answer, model)

Хуки SAE:

- `hook_sae_input` — вход SAE, активация модели, `[batch, seq_len, d_in]`;
- `hook_sae_acts_pre` — $xW_{enc} + b_{enc}$ до ReLU, `[batch, seq_len, d_sae]`;
- `hook_sae_acts_post` — признаки после ReLU, их и читают, `[batch, seq_len, d_sae]`;
- `hook_sae_recons` — реконструкция $f W_{dec} + b_{dec}$, `[batch, seq_len, d_in]`;
- `hook_sae_error` — ошибка реконструкции; считается, только если `sae.use_error_term = True`;
- `hook_sae_output` — то, что SAE передает дальше по модели.

По умолчанию `use_error_term = False`: дальше по модели идет реконструкция, а не исходная активация.

In [ ]:
_, cache = model.run_with_cache_with_saes(prompt, saes=[sae])

# Ваш код здесь: выведите имена и размеры всех записей кэша, в имени которых есть "sae"

**Задание 10. Сколько записей кэша связаны с SAE?**

In [ ]:
acts = cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0]  # [позиция, d_sae]

px.line(acts[-1].cpu().numpy(), title="Feature activations at the final token position",
        labels={"index": "Feature", "value": "Activation"}).show()

vals, inds = torch.topk(acts[-1], 5)
for val, ind in zip(vals, inds):
    print(f"Feature {int(ind)} fired {val:.2f}")

**Задание 11. Какой номер у признака с наибольшей активацией на последнем токене?**

Признаков про кошек среди пяти самых сильных нет, хотя модель ставит « cat» третьим вариантом. Вспомните урок о logit lens: на слое 7 из 12 модель еще не собрала ответ.

In [ ]:
l0 = ...  # Ваш код здесь: число ненулевых признаков на каждой позиции промпта
print("L0 по позициям:", l0)

**Задание 12. Чему равен $L_0$ на последнем токене промпта — сколько признаков из 24 576 ненулевые?**

In [ ]:
x = cache["blocks.7.hook_resid_pre.hook_sae_input"][0, 1:]  # без первой позиции <|endoftext|>
x_hat = cache["blocks.7.hook_resid_pre.hook_sae_recons"][0, 1:]

fve = ...  # Ваш код здесь: 1 - сумма квадратов ошибки / сумма квадратов отклонений x от среднего по позициям
print("доля объясненной дисперсии:", fve)

**Задание 13. Какую долю дисперсии активаций объясняет реконструкция SAE на этом промпте? Ответ округлите до сотых.**

Наконец, подставим реконструкцию SAE вместо активации слоя 7 и посмотрим, что станет с прогнозом. Расхождение распределений измерим KL-дивергенцией.

In [ ]:
clean_logits = model(prompt)[0, -1]
spliced_logits = model.run_with_saes(prompt, saes=[sae])[0, -1]  # вместо активации слоя 7 идет реконструкция SAE

for name, logits in [("без SAE", clean_logits), ("с реконструкцией SAE", spliced_logits)]:
    print(name, [model.tokenizer.decode(t) for t in logits.topk(3).indices])

p, q = clean_logits.softmax(-1), spliced_logits.softmax(-1)
print("KL(без SAE || с SAE):", round((p * (p.log() - q.log())).sum().item(), 3))

**Задание 14. Что изменилось в прогнозе модели, когда активацию слоя 7 заменили реконструкцией SAE?**

## **Заключение**

- Игрушечная модель показывает суперпозицию в чистом виде: при плотных признаках модель хранит столько признаков, сколько у нее измерений, и отбрасывает остальные; при разреженных складывает больше признаков, чем измерений, в почти ортогональные направления и гасит помехи отрицательным смещением.
- SAE пытается развернуть такую упаковку обратно: словарь в 32 раза шире активации, а на одном токене ненулевых признаков несколько десятков.
- Реконструкция SAE неточна: она объясняет не всю дисперсию активаций и немного меняет прогноз модели. Поэтому вывод «модель пользуется признаком» требует вмешательства, а не только чтения признака.
- Описания признаков — гипотезы модели-аннотатора; их проверяют на новых текстах.